# 📊 01 — Exploratory Data Analysis (EDA)

**Objectif :** Comprendre les données avant de construire quoi que ce soit.

Règle d'or en ML : **toujours explorer les données avant de coder le modèle.**
Les surprises dans les données (déséquilibre, valeurs manquantes, bruit) doivent être
détectées maintenant, pas après des heures d'entraînement.

**Ce notebook répond à :**
- Combien de messages ? Quelle répartition ham/spam ?
- Les messages spam sont-ils plus longs que les ham ?
- Quels mots sont les plus fréquents dans les spams ? Dans les ham ?
- Y a-t-il des valeurs manquantes ?
- Le dataset est-il déséquilibré ?

## 0. Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter

# Style visuel
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'

print('✅ Imports OK')

## 1. Chargement des données

In [ ]:
# Chemin relatif depuis le dossier notebooks/
DATA_PATH = '../data/sms_spam.csv'

df = pd.read_csv(
    DATA_PATH,
    sep='\t',           # Séparateur tabulation
    header=None,        # Pas d'en-tête dans le fichier
    names=['label', 'text'],  # On nomme les colonnes nous-mêmes
    encoding='latin-1', # Encodage du dataset UCI
)

print(f'Shape : {df.shape}')  # (nb_lignes, nb_colonnes)
df.head(10)

In [ ]:
# Vérification des types et valeurs manquantes
print('=== Informations générales ===')
df.info()

print('\n=== Valeurs manquantes ===')
print(df.isnull().sum())

print('\n=== Doublons ===')
print(f'Lignes dupliquées : {df.duplicated().sum()}')

In [ ]:
# Encodage binaire du label pour faciliter les analyses
# 'ham'  → 0  (message légitime)
# 'spam' → 1  (message indésirable)
df['label_num'] = df['label'].map({'ham': 0, 'spam': 1})

print('Distribution des labels :')
print(df['label'].value_counts())
print(f"\nTaux de spam : {df['label_num'].mean()*100:.1f}%")

## 2. Distribution des classes

**Question clé :** Le dataset est-il équilibré ?

Si le spam représente 13% des données, un modèle qui prédit *toujours ham*
aurait 87% d'accuracy — sans rien apprendre. C'est pourquoi on utilise le **F1-Score**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# ── Graphique 1 : Barplot ─────────────────────────────────────────────────────
counts = df['label'].value_counts()
colors = ['#4CAF50', '#F44336']  # vert = ham, rouge = spam

bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)

# Affiche les valeurs et pourcentages sur chaque barre
for bar, (label, count) in zip(bars, counts.items()):
    pct = count / len(df) * 100
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 20,
        f'{count}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=12, fontweight='bold'
    )

axes[0].set_title('Distribution Ham vs Spam', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Nombre de messages')
axes[0].set_ylim(0, counts.max() * 1.2)

# ── Graphique 2 : Camembert ───────────────────────────────────────────────────
axes[1].pie(
    counts.values,
    labels=[f'Ham\n{counts["ham"]} ({counts["ham"]/len(df)*100:.1f}%)',
            f'Spam\n{counts["spam"]} ({counts["spam"]/len(df)*100:.1f}%)'],
    colors=colors,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
)
axes[1].set_title('Proportion Ham vs Spam', fontsize=13, fontweight='bold')

plt.suptitle('Dataset SMS Spam Collection — Équilibre des classes',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"\n⚠️  Ratio Ham/Spam : {counts['ham']/counts['spam']:.1f}:1")
print("→ Dataset déséquilibré : l'Accuracy seule n'est pas une bonne métrique.")
print("→ On utilisera le F1-Score pour évaluer les modèles.")

## 3. Analyse de la longueur des messages

**Hypothèse :** Les spams contiennent souvent plus de texte (longues offres promotionnelles)
que les ham (messages courts du quotidien).

In [ ]:
# Feature engineering basique : longueur en caractères et en mots
df['char_count']  = df['text'].str.len()
df['word_count']  = df['text'].str.split().str.len()

# Statistiques par classe
print('=== Longueur en caractères ===')
print(df.groupby('label')[['char_count', 'word_count']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ham_data  = df[df['label'] == 'ham']
spam_data = df[df['label'] == 'spam']

# ── Distribution de la longueur en caractères ─────────────────────────────────
axes[0].hist(ham_data['char_count'],  bins=50, alpha=0.7, color='#4CAF50', label='Ham',  edgecolor='white')
axes[0].hist(spam_data['char_count'], bins=50, alpha=0.7, color='#F44336', label='Spam', edgecolor='white')
axes[0].axvline(ham_data['char_count'].mean(),  color='#2E7D32', linestyle='--', linewidth=2, label=f'Moy Ham = {ham_data["char_count"].mean():.0f}')
axes[0].axvline(spam_data['char_count'].mean(), color='#B71C1C', linestyle='--', linewidth=2, label=f'Moy Spam = {spam_data["char_count"].mean():.0f}')
axes[0].set_title('Distribution longueur (caractères)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Nombre de caractères')
axes[0].set_ylabel('Fréquence')
axes[0].legend()

# ── Boxplot longueur en mots ──────────────────────────────────────────────────
plot_data = [
    {'label': 'Ham',  'word_count': ham_data['word_count']},
    {'label': 'Spam', 'word_count': spam_data['word_count']},
]
bp = axes[1].boxplot(
    [ham_data['word_count'], spam_data['word_count']],
    labels=['Ham', 'Spam'],
    patch_artist=True,
    boxprops=dict(facecolor='#E8F5E9'),
    medianprops=dict(color='#1B5E20', linewidth=2),
)
bp['boxes'][1].set_facecolor('#FFEBEE')
axes[1].set_title('Distribution longueur (mots)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Nombre de mots')

plt.suptitle('Longueur des messages : Ham vs Spam', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_message_length.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Longueur moyenne Ham  : {ham_data['char_count'].mean():.0f} caractères")
print(f"Longueur moyenne Spam : {spam_data['char_count'].mean():.0f} caractères")
print(f"→ Les spams sont {spam_data['char_count'].mean()/ham_data['char_count'].mean():.1f}× plus longs en moyenne")

## 4. Mots les plus fréquents

**Objectif :** Identifier les mots discriminants — ceux qui apparaissent
massivement dans les spams mais rarement dans les ham, et vice-versa.

In [ ]:
from collections import Counter

def get_top_words(series, n=20):
    """
    Retourne les N mots les plus fréquents d'une Series de textes.
    
    Explications :
    - series.str.lower() → minuscules
    - .str.split()       → chaque texte devient une liste de mots
    - .explode()         → transforme les listes en lignes individuelles
    - Counter            → compte les occurrences de chaque mot
    """
    all_words = series.str.lower().str.split().explode()
    # Filtre les mots très courts (bruit)
    all_words = all_words[all_words.str.len() > 2]
    return Counter(all_words).most_common(n)

ham_top  = get_top_words(ham_data['text'],  n=20)
spam_top = get_top_words(spam_data['text'], n=20)

print('Top 10 mots dans les HAM :')
for word, count in ham_top[:10]:
    print(f'  {word:<15} : {count}')

print('\nTop 10 mots dans les SPAM :')
for word, count in spam_top[:10]:
    print(f'  {word:<15} : {count}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, top_words, title, color in zip(
    axes,
    [ham_top, spam_top],
    ['Top 20 mots — HAM', 'Top 20 mots — SPAM'],
    ['#4CAF50', '#F44336']
):
    words  = [w for w, _ in top_words]
    counts = [c for _, c in top_words]
    
    bars = ax.barh(words[::-1], counts[::-1], color=color, alpha=0.85, edgecolor='white')
    
    for bar, count in zip(bars, counts[::-1]):
        ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
                str(count), va='center', fontsize=8)
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Fréquence')

plt.suptitle('Mots les plus fréquents avant prétraitement', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_top_words.png', bbox_inches='tight', dpi=150)
plt.show()

print("\n💡 Observation : beaucoup de stopwords ('to', 'the', 'a'...)")
print("   → Le prétraitement les supprimera pour ne garder que les mots porteurs de sens.")

## 5. WordCloud

Un nuage de mots donne une vision visuelle et intuitive des termes dominants.
Plus un mot est grand, plus il est fréquent.

In [ ]:
os.makedirs('../reports', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, subset, title, colormap in zip(
    axes,
    [ham_data, spam_data],
    ['HAM — Messages légitimes', 'SPAM — Messages indésirables'],
    ['Greens', 'Reds']
):
    # Concatène tous les textes en une seule chaîne
    all_text = ' '.join(subset['text'].str.lower().values)
    
    wc = WordCloud(
        width=800,
        height=400,
        background_color='white',
        colormap=colormap,
        max_words=100,
        # stopwords intégrés dans WordCloud pour éviter les mots vides visuels
        stopwords=None,
    ).generate(all_text)
    
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')  # Pas d'axes pour les wordclouds
    ax.set_title(title, fontsize=13, fontweight='bold', pad=10)

plt.suptitle('WordCloud — Ham vs Spam (avant prétraitement)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_wordcloud.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. Synthèse EDA

**Ce qu'on a appris :**

In [ ]:
print('=' * 60)
print('SYNTHÈSE EDA — SMS SPAM COLLECTION')
print('=' * 60)

print(f"""
📊 Dataset
   Total messages  : {len(df)}
   Ham (légitimes) : {len(ham_data)} ({len(ham_data)/len(df)*100:.1f}%)
   Spam            : {len(spam_data)} ({len(spam_data)/len(df)*100:.1f}%)

⚠️  Déséquilibre
   Ratio Ham/Spam = {len(ham_data)/len(spam_data):.1f}:1
   → Accuracy seule trompeuse, utiliser F1-Score

📏 Longueur
   Ham  : {ham_data['char_count'].mean():.0f} caractères en moyenne
   Spam : {spam_data['char_count'].mean():.0f} caractères en moyenne
   → La longueur seule pourrait discriminer ham/spam à {spam_data['char_count'].mean()/ham_data['char_count'].mean()*100-100:.0f}% de différence

🔤 Vocabulaire
   Mots spam typiques : free, call, txt, win, prize, claim, urgent
   Mots ham typiques  : go, get, ok, know, come, time, got

✅ Données
   Valeurs manquantes : {df.isnull().sum().sum()}
   Doublons           : {df.duplicated().sum()}
   → Dataset propre, prêt pour le prétraitement
""")

print('💡 Prochaine étape : notebook 02_classical_nlp.ipynb')